# Notebook for transcribing audio using Google Cloud Speech-to-Text

### imports

In [1]:
import quail
import os
import shutil
import pickle
import re
import difflib
import string
import pandas as pd
from nltk.corpus import stopwords
from num2words import num2words

### set some paths

In [2]:
audiodir = os.path.abspath('../../../data/audio/')
transcdir = os.path.abspath('../../../data/transcriptions/automatic/')
keypath = os.path.abspath('../../../../google-credentials/cloud-speech-credentials.json')
annotdir = os.path.abspath('../../../data/annotations_dfs/')
pickledir = os.path.abspath('../../../data/pickles/')

## create speech context from union of words in annotations

In [3]:
atlep1_df = pd.read_pickle(annotdir+'/atlep1.p')
atlep2_df = pd.read_pickle(annotdir+'/atlep2.p')
arrdev_df = pd.read_pickle(annotdir+'/arrdev.p')

In [4]:
def meets_word_criteria(string):
    """
    Removes words with characters not wanted in auto transcriber speech context
    """
    
    good_word = True
    
    # remove words containing digits
    if any(char.isdigit() for char in string):
        good_word = False
    
    # remove words surrounded by single quotes and possessives (avoid duplicates in nested quotations & possessives)
    if string.startswith("'") or string.endswith("'") or string.endswith("'s"):
        good_word = False
        
    # remove unhelpful simple words
    if len(string) <= 2:
        good_word = False
    
    return good_word

In [5]:
def create_speech_context(df):
    """
    Creates episode-specific speech context from video annotations
    """
    
    # use Narrative details (internal and external), Characters on screen, Speech, Character speaking, and Setting
    word_cols = [df.columns[i] for i in [2,3,4,6,7,9]]
    
    # create single string of all text
    allwords = ' '.join(df.loc[:,word_cols].apply(lambda x: ' '.join(x.dropna()), axis=1).values.tolist())
    
    # remove all characters except spaces (catches \n and \t), letters, apostrophes, dashes
    no_punctuation = re.sub("[^\w\s'-]+", '', allwords)
    
    speech_context = []
    
    # split words into list
    for word in no_punctuation.split():
        # identify unique words that meet criteria
        if word.upper() not in speech_context and meets_word_criteria(word):
            speech_context.append(word.upper())

    # remove English stopwords
    speech_context_nostop = [word for word in speech_context if word not in stopwords.words('english')]
    
    return speech_context_nostop

In [6]:
atlep1_speech_context = create_speech_context(atlep1_df)
atlep2_speech_context = create_speech_context(atlep2_df)
arrdev_speech_context = create_speech_context(arrdev_df)

### load in experiment data and mapings between subject ID & PsiTurk ID

In [7]:
with open(pickledir+'/expdf.p', 'rb') as f:
    expdf = pickle.load(f)

with open(pickledir+'/id_maps.p', 'rb') as f:
    id_maps = pickle.load(f)

### create directory structure

In [8]:
for sid, data in id_maps.items():
    for ses, turkid in data.items():
        folder = os.path.join(transcdir,sid,turkid)
        if not os.path.isdir(folder) and not os.path.isdir(os.path.join(transcdir,'drops',sid)):
            os.makedirs(folder)

## transcribe all audio files not previously transcribed

In [9]:
turkids = [tid for l in [list(ses.values()) for ses in id_maps.values()] for tid in l if tid]

done = False

# walk audio folder
for root, dirs, files in os.walk(audiodir):
    
    # ignore parent dirs with hiden files
    if [f for f in files if not f.startswith('.')]:
        # assign psiturk id
        turkid = files[0].split('-')[0]
        # ignore drops
        if turkid in turkids:
            # assign subject id
            sid = expdf.loc[expdf.uniqueid==turkid]['Subject ID'].values[0]
            audio_files = [file for file in files if file.endswith('wav')]
            
            for audio_file in audio_files:
                
                # set correct speech context
                if (any([af.split('-')[1].startswith('prediction') for af in audio_files]) 
                    or audio_file.split('-')[1].startswith('delayed')):
                    speech_context = atlep1_speech_context
                elif 'A' in sid:
                    speech_context = atlep2_speech_context
                else:
                    speech_context = arrdev_speech_context
                
                af_path = os.path.join(root,audio_file)
                save_dir = os.path.join(transcdir,sid,turkid)
                
                # skip over previously decoded audio
                if not os.path.isfile(os.path.join(save_dir,audio_file+'.txt')):
                    
                # decode audio file and save in specified dir
                    print('decoding ' + audio_file)
                    quail.decode_speech(af_path, keypath=keypath, save=True, save_dir=save_dir,
                                        speech_context=speech_context, max_alternatives=5)
                
                else:
                    print('already finished '+ audio_file)

decoding debugvnS2Q:debugG20gK-prediction.wav
Decoding file 1 of 1
Audio clip is longer than 1 minute.  Splitting into 6 one minute segments...


KeyboardInterrupt: 

In [15]:
audio_file.strip('wav')

'debugvnS2Q:debugG20gK-prediction.wav'

## find/replace transcription instances to format for modeling

In [205]:
# # reaname initial output files to keep separate
# for root, dirs, files in os.walk(transcdir):
#     transcripts = [f for f in files if f.endswith('.wav.txt')]
#     if transcripts:
#         for tr in transcripts:
#             oldname = os.path.join(root,tr)
#             fname, ext = oldname.split('.', maxsplit=1)
#             newname = fname+'-raw.'+ext
#             os.rename(oldname, newname)

In [13]:
# define some mappings

replacement_words = {
    'PAPERBOYS' : "PAPER BOY'S",
    'PAPERBOY' : 'PAPER BOY',
    'URN' : 'EARN',
    'EARNS' : "EARN'S",
    'URNS' : "EARN'S"
}

replacement_chars = {
    '.' : '',
    ',' : '',
    '!' : '',
    '?' : '',
    '"' : '',
    '/' : 'SLASH',
    '&' : ' AND ',
    '+' : 'PLUS',
    '%' : ' PERCENT'
}

def replace_func(word, replacement_dict):
    return re.sub('({})'.format('|'.join(map(re.escape, replacement_dict.keys()))), 
                  lambda m: replacement_dict[m.group()], word)

In [17]:
for root, dirs, files in os.walk(transcdir):
    transcripts = [f for f in files if f.endswith('.wav.txt')]
    if transcripts:
        for tr in transcripts:
            transc = [line.strip('\n') for line in open(os.path.join(root,tr), 'r')]
            

            for ix, line in enumerate(transc):
                
                # remove commas from transcription
                if len(transc[ix].split(',')) > 3:
                    transc[ix] = transc[ix].replace(',','', 1)                
                
                # remove unwanted punctuation
                if any(str(k) in transc[ix] for k in replacement_chars.keys()):
                    transc[ix] = transc[ix].replace(transc[ix].split(',')[0], replace_func(transc[ix].split(',')[0], 
                                                                                           replacement_chars))

                
                # catch mistranscriptions of words
                if transc[ix].split(',')[0] in replacement_words.keys():
                    transc[ix] = transc[ix].replace(transc[ix].split(',')[0], replace_func(transc[ix].split(',')[0],
                                                                                           replacement_words))
                
                # catch incorrect splitting of of character name
                if transc[ix].split(',')[0] == 'GEORGE' and transc[ix+1].split(',')[0] == 'MICHAEL':
                    transc[ix] = transc[ix].replace('GEORGE', 'GEORGE-MICHAEL')
                    del transc[ix+1]
                    
                # convert digits to words
                if transc[ix].split(',')[0].isdigit():
                    transc[ix] = transc[ix].replace(transc[ix].split(',')[0],
                                                    num2words(int(transc[ix].split(',')[0])).upper())
                
                # convert times to words
                if ':' in transc[ix].split(',')[0]:
                    newword = ' '.join([num2words(int(i)) for i in transc[ix].split(',')[0].split(':')])
                    transc[ix] = transc[ix].replace(transc[ix].split(',')[0], newword.upper())
                    
                # convert currency to words
                if '$' in transc[ix].split(',')[0]:
                    newword = num2words(transc[ix].split(',')[0].strip('$'), to='currency',
                                        currency='USD').split(',')[0]
                    transc[ix] = transc[ix].replace(transc[ix].split(',')[0], newword.upper())
                    
                
                # remove reamining empty strings
                if not transc[ix].split(',')[0]:
                    del transc[ix]
                    
            out_name = os.path.join(root, tr.replace('raw','corrected'))
            
            with open(out_name, 'w') as f:
                for line in transc:
                    f.write(line + '\n')

In [74]:
with open(transcdir+'/MD-013119-B-01/debuguhHBa:debug1HOCZ/debuguhHBa:debug1HOCZ-recall.wav.p', 'rb') as f:
    results_obj = pickle.load(f)

In [121]:
# iterate over nested levels of Cloud Speech RecognizeResponse objects
for chunk in results_obj:
    for res in chunk.results:
        # find transcript alternative with max confidence
        max_conf = max([alt.confidence for alt in res.alternatives])
        for ix, alt in enumerate(res.alternatives):
            # if it's not the chosen alternative
            if alt.confidence == max_conf and ix != 0:
                best_alt = alt.transcript
                first_alt = res.alternatives[0].transcript
        break
    break

In [117]:
best_alt

'so the episode started off in front of a liquor store and he saw like a white car in front of it and The Paperboy song was playing on the radio and then all of a sudden you start hearing this one guy in the car screen and then earn like I got out of the car and started screaming his name Alfred and then they went up and started talking to this one guy in this girl and Alfred pulled a gun on the guy ever said he wanted money for the mirror that he kicked off his car and then so he pulled his gun out and then earn hold his out and then the guy that they were trying to get money from'

In [122]:
first_alt

'so the episode started off in front of a liquor store and he saw like a white car in front of it and The Paperboy song was playing on the radio and then all of a sudden you start hearing this one guy in the car screen and then earned my got out of the car and started screaming his name Alfred and then they went up and started talking to this one guy in this girl and Alfred pulled a gun on the guy ever said he wanted money for the mirror that he kicked off his car and then so he pulled his gun out and then earn hold his out and then the guy that they were trying to get money from'

In [139]:
[i for i in difflib.ndiff(first_alt.split(' '), best_alt.split(' '))]

['  so',
 '  the',
 '  episode',
 '  started',
 '  off',
 '  in',
 '  front',
 '  of',
 '  a',
 '  liquor',
 '  store',
 '  and',
 '  he',
 '  saw',
 '  like',
 '  a',
 '  white',
 '  car',
 '  in',
 '  front',
 '  of',
 '  it',
 '  and',
 '  The',
 '  Paperboy',
 '  song',
 '  was',
 '  playing',
 '  on',
 '  the',
 '  radio',
 '  and',
 '  then',
 '  all',
 '  of',
 '  a',
 '  sudden',
 '  you',
 '  start',
 '  hearing',
 '  this',
 '  one',
 '  guy',
 '  in',
 '  the',
 '  car',
 '  screen',
 '  and',
 '  then',
 '- earned',
 '?     --\n',
 '+ earn',
 '- my',
 '+ like',
 '+ I',
 '  got',
 '  out',
 '  of',
 '  the',
 '  car',
 '  and',
 '  started',
 '  screaming',
 '  his',
 '  name',
 '  Alfred',
 '  and',
 '  then',
 '  they',
 '  went',
 '  up',
 '  and',
 '  started',
 '  talking',
 '  to',
 '  this',
 '  one',
 '  guy',
 '  in',
 '  this',
 '  girl',
 '  and',
 '  Alfred',
 '  pulled',
 '  a',
 '  gun',
 '  on',
 '  the',
 '  guy',
 '  ever',
 '  said',
 '  he',
 '  wanted',
 

In [145]:
difflib.get_close_matches('paper boy', ['hello', 'goodbye', 'help', 'paperboy'])

['paperboy']

In [144]:
a[51]

'+ earn'

In [222]:
'asdjfkl,1.4,5.5'.replace('a','b')

'bsdjfkl,1.4,5.5'

In [75]:
for i in results_obj[0].results[1].alternatives:
    print(i.transcript)
    print(i.confidence)

show that he had a gun at the same time and
0.9578661918640137
so that he had a gun at the same time and
0.9581863284111023
saw that he had a gun at the same time and
0.9116089344024658
show that he had a gun at same time and
0.9250459671020508
sure that he had a gun at the same time and
0.9466335773468018
